# Join Team Qualities to Transfer Data

Merge `team_qualities.parquet` into `within_league_transfers_v4.parquet`.

**Join logic:**
- All transfers are within-league with `to_season = from_season + 1`
- **from_team** → **projected** qualities (`q_proj_`) + **projected** z-scores (`z_proj_*`)
- **to_team** → **current** qualities (`q_`) + **current** z-scores (`z_*`)

Both sides are expressed relative to the **same season's league population** (to_season), making them directly comparable.

**Also:**
- **Drops** 148 raw team stats (`from_team_stats_*`, `to_team_stats_*`) — redundant now that we have qualities and z-scores
- **Keeps** the 21 quality-metric z-scores per side in case they add value to the model

**Input:** `within_league_transfers_v4.parquet`, `team_qualities.parquet`  
**Output:** `within_league_transfers_v5.parquet`

In [7]:
import pandas as pd
import numpy as np
import os

pd.set_option("display.max_columns", 40)
pd.set_option("display.float_format", "{:.4f}".format)

DATA_DIR = "../../../thesis_data/processed_data/thesis_model_dataset"
V4_PATH = f"{DATA_DIR}/within_league_transfers_v4.parquet"
TQ_PATH = f"{DATA_DIR}/team_qualities.parquet"
OUTPUT_PATH = f"{DATA_DIR}/within_league_transfers_v5.parquet"

v4 = pd.read_parquet(V4_PATH)
tq = pd.read_parquet(TQ_PATH)

print(f"v4 transfers: {v4.shape[0]:,} rows x {v4.shape[1]} columns")
print(f"team qualities: {tq.shape[0]:,} rows x {tq.shape[1]} columns")

v4 transfers: 18,065 rows x 210 columns
team qualities: 13,817 rows x 83 columns


## 1. Prepare team quality columns for join

In [8]:
# Identify column groups in team_qualities parquet
q_current_cols = [c for c in tq.columns if c.startswith("q_") and not c.startswith("q_proj_")]
q_proj_cols = [c for c in tq.columns if c.startswith("q_proj_")]
z_current_cols = [c for c in tq.columns if c.startswith("z_") and not c.startswith("z_proj_")]
z_proj_cols = [c for c in tq.columns if c.startswith("z_proj_")]

print(f"Current qualities:    {q_current_cols}")
print(f"Projected qualities:  {q_proj_cols}")
print(f"Current z-scores:     {len(z_current_cols)} cols")
print(f"Projected z-scores:   {len(z_proj_cols)} cols")

# For from_team: projected qualities + projected z-scores
from_join_cols = q_proj_cols + z_proj_cols
from_tq = tq[["team_id", "competition_id", "season"] + from_join_cols].copy()
from_tq = from_tq.rename(columns={
    "team_id": "from_team_id",
    "competition_id": "from_competition",
    "season": "from_season",
    **{c: f"from_{c}" for c in from_join_cols},
})

# For to_team: current qualities + current z-scores
to_join_cols = q_current_cols + z_current_cols
to_tq = tq[["team_id", "competition_id", "season"] + to_join_cols].copy()
to_tq = to_tq.rename(columns={
    "team_id": "to_team_id",
    "competition_id": "to_competition",
    "season": "to_season",
    **{c: f"to_{c}" for c in to_join_cols},
})

print(f"\nfrom_tq: {from_tq.shape[1]} cols (3 keys + {len(q_proj_cols)} qualities + {len(z_proj_cols)} z-scores)")
print(f"to_tq:   {to_tq.shape[1]} cols (3 keys + {len(q_current_cols)} qualities + {len(z_current_cols)} z-scores)")

Current qualities:    ['q_DEFENCE', 'q_DEFENSIVE_TRANSITION', 'q_ATTACKING_TRANSITION', 'q_ATTACK', 'q_PENETRATION', 'q_CHANCE_CREATION', 'q_OUTCOME']
Projected qualities:  ['q_proj_DEFENCE', 'q_proj_DEFENSIVE_TRANSITION', 'q_proj_ATTACKING_TRANSITION', 'q_proj_ATTACK', 'q_proj_PENETRATION', 'q_proj_CHANCE_CREATION', 'q_proj_OUTCOME']
Current z-scores:     22 cols
Projected z-scores:   22 cols

from_tq: 32 cols (3 keys + 7 qualities + 22 z-scores)
to_tq:   32 cols (3 keys + 7 qualities + 22 z-scores)


## 2. Join to v4

In [9]:
n_before = len(v4)

# Drop raw team stats from v4 (replaced by z-scores + qualities)
raw_ts_cols = [c for c in v4.columns if c.startswith("from_team_stats_") or c.startswith("to_team_stats_")]
v5 = v4.drop(columns=raw_ts_cols)
print(f"Dropped {len(raw_ts_cols)} raw team stats columns from v4")

# Join from_team projected qualities + z-scores
v5 = v5.merge(
    from_tq,
    on=["from_team_id", "from_competition", "from_season"],
    how="left",
    validate="m:1",
)

# Join to_team current qualities + z-scores
v5 = v5.merge(
    to_tq,
    on=["to_team_id", "to_competition", "to_season"],
    how="left",
    validate="m:1",
)

assert len(v5) == n_before, f"Row count changed: {n_before} → {len(v5)}"
print(f"\nRows: {len(v5):,} (unchanged ✓)")
print(f"Columns: {v4.shape[1]} → {v5.shape[1]} ({v5.shape[1] - v4.shape[1]:+d} net: -{len(raw_ts_cols)} raw stats, +{len(from_join_cols)+len(to_join_cols)} qualities & z-scores)")

Dropped 148 raw team stats columns from v4

Rows: 18,065 (unchanged ✓)
Columns: 210 → 120 (-90 net: -148 raw stats, +58 qualities & z-scores)


## 3. Validate join

In [10]:
# Check nulls in the new columns
new_from_q = [c for c in v5.columns if c.startswith("from_q_proj_")]
new_from_z = [c for c in v5.columns if c.startswith("from_z_proj_")]
new_to_q = [c for c in v5.columns if c.startswith("to_q_") and not c.startswith("to_q_proj")]
new_to_z = [c for c in v5.columns if c.startswith("to_z_") and not c.startswith("to_z_proj")]

print("Null check — from_team PROJECTED qualities:")
for c in new_from_q:
    n_null = v5[c].isnull().sum()
    print(f"  {c:45s} nulls={n_null:,} ({100*n_null/len(v5):.1f}%)")

print(f"\nNull check — from_team PROJECTED z-scores ({len(new_from_z)} cols):")
from_z_nulls = v5[new_from_z].isnull().sum()
print(f"  min nulls={from_z_nulls.min():,}  max nulls={from_z_nulls.max():,}  ({100*from_z_nulls.max()/len(v5):.1f}%)")

print(f"\nNull check — to_team CURRENT qualities:")
for c in new_to_q:
    n_null = v5[c].isnull().sum()
    print(f"  {c:45s} nulls={n_null:,} ({100*n_null/len(v5):.1f}%)")

print(f"\nNull check — to_team CURRENT z-scores ({len(new_to_z)} cols):")
to_z_nulls = v5[new_to_z].isnull().sum()
print(f"  min nulls={to_z_nulls.min():,}  max nulls={to_z_nulls.max():,}  ({100*to_z_nulls.max()/len(v5):.1f}%)")

Null check — from_team PROJECTED qualities:
  from_q_proj_DEFENCE                           nulls=881 (4.9%)
  from_q_proj_DEFENSIVE_TRANSITION              nulls=881 (4.9%)
  from_q_proj_ATTACKING_TRANSITION              nulls=881 (4.9%)
  from_q_proj_ATTACK                            nulls=881 (4.9%)
  from_q_proj_PENETRATION                       nulls=881 (4.9%)
  from_q_proj_CHANCE_CREATION                   nulls=881 (4.9%)
  from_q_proj_OUTCOME                           nulls=881 (4.9%)

Null check — from_team PROJECTED z-scores (22 cols):
  min nulls=881  max nulls=883  (4.9%)

Null check — to_team CURRENT qualities:
  to_q_DEFENCE                                  nulls=610 (3.4%)
  to_q_DEFENSIVE_TRANSITION                     nulls=610 (3.4%)
  to_q_ATTACKING_TRANSITION                     nulls=610 (3.4%)
  to_q_ATTACK                                   nulls=610 (3.4%)
  to_q_PENETRATION                              nulls=610 (3.4%)
  to_q_CHANCE_CREATION                    

In [11]:
# Investigate null rows if any
all_new_cols = new_from_q + new_from_z + new_to_q + new_to_z
any_null = v5[all_new_cols].isnull().any(axis=1)
from_null_mask = v5[new_from_q + new_from_z].isnull().any(axis=1)
to_null_mask = v5[new_to_q + new_to_z].isnull().any(axis=1)

print(f"Transfers with ANY null team quality/z-score: {any_null.sum():,} / {len(v5):,}")
print(f"  From-side null: {from_null_mask.sum():,}")
print(f"  To-side null:   {to_null_mask.sum():,}")

if any_null.sum() > 0:
    print(f"\nNull rows by from_season:")
    print(v5.loc[from_null_mask, "from_season"].value_counts().sort_index().to_string())
    print(f"\nNull rows by competition (top 10):")
    print(v5.loc[any_null, "from_competition"].value_counts().head(10).to_string())

Transfers with ANY null team quality/z-score: 1,091 / 18,065
  From-side null: 884
  To-side null:   617

Null rows by from_season:
from_season
2018    171
2019    266
2020    260
2021     34
2022     21
2023    122
2024     10

Null rows by competition (top 10):
from_competition
516     95
796     77
535     63
725     46
286     38
534     37
287     37
520     35
551     32
1710    30


In [12]:
# Quick stats on the new quality columns
print("from_team PROJECTED qualities (summary):")
print(v5[new_from_q].describe().round(3).to_string())
print(f"\nto_team CURRENT qualities (summary):")
print(v5[new_to_q].describe().round(3).to_string())

from_team PROJECTED qualities (summary):
       from_q_proj_DEFENCE  from_q_proj_DEFENSIVE_TRANSITION  from_q_proj_ATTACKING_TRANSITION  from_q_proj_ATTACK  from_q_proj_PENETRATION  from_q_proj_CHANCE_CREATION  from_q_proj_OUTCOME
count           17184.0000                        17184.0000                        17184.0000          17184.0000               17184.0000                   17184.0000           17184.0000
mean                0.0210                            0.0290                            0.0310             -0.1030                  -0.0730                      -0.0130              -0.0970
std                 0.9470                            0.7870                            0.5490              0.6700                   0.6220                       0.5760               0.9770
min                -6.8720                          -15.6920                           -4.0710             -3.6420                  -3.6140                      -4.2230              -5.3620
25%      

In [13]:
# Sanity: from projected and to current should be on similar scales
# (both are z-score-based, normalised by total weight)
print("Scale comparison — mean(abs(quality)):")
for q in ["DEFENCE", "DEFENSIVE_TRANSITION", "ATTACKING_TRANSITION", "ATTACK",
          "PENETRATION", "CHANCE_CREATION", "OUTCOME"]:
    f_col = f"from_q_proj_{q}"
    t_col = f"to_q_{q}"
    f_abs = v5[f_col].abs().mean()
    t_abs = v5[t_col].abs().mean()
    print(f"  {q:25s}  from_proj={f_abs:.4f}  to_curr={t_abs:.4f}")

Scale comparison — mean(abs(quality)):
  DEFENCE                    from_proj=0.7178  to_curr=0.6350
  DEFENSIVE_TRANSITION       from_proj=0.5655  to_curr=0.4413
  ATTACKING_TRANSITION       from_proj=0.4025  to_curr=0.3236
  ATTACK                     from_proj=0.5203  to_curr=0.4526
  PENETRATION                from_proj=0.4663  to_curr=0.3770
  CHANCE_CREATION            from_proj=0.4069  to_curr=0.3157
  OUTCOME                    from_proj=0.7618  to_curr=0.7236


## 4. Final column overview

In [14]:
# Column groups in v5
cols = v5.columns.tolist()

TWELVE_QS_NAMES = [
    "Active defence", "Aerial threat", "Box threat", "Chance prevention",
    "Composure", "Defensive heading", "Dribbling", "Effectiveness",
    "Finishing", "Hold-up play", "Intelligent defence", "Involvement",
    "Passing quality", "Poaching", "Pressing", "Progression",
    "Providing teammates", "Run quality", "Territorial dominance", "Winning duels",
]
twelve_qs_set = set()
for side in ["from", "to"]:
    for n in TWELVE_QS_NAMES:
        twelve_qs_set.add(f"{side}_{n}")

twelve_qs = [c for c in cols if c in twelve_qs_set]
team_q_from = [c for c in cols if c.startswith("from_q_proj_")]
team_q_to = [c for c in cols if c.startswith("to_q_") and "z_" not in c]
team_z_from = [c for c in cols if c.startswith("from_z_proj_")]
team_z_to = [c for c in cols if c.startswith("to_z_") and not c.startswith("to_z_proj")]
structural = [c for c in cols if c not in set(twelve_qs + team_q_from + team_q_to + team_z_from + team_z_to)]

print(f"v5 shape: {v5.shape[0]:,} rows x {v5.shape[1]} columns")
print(f"\nColumn groups:")
print(f"  Structural/ID:                     {len(structural)}")
print(f"  Twelve Player QS (from+to):        {len(twelve_qs)}")
print(f"  Team qualities — from projected:   {len(team_q_from)}")
print(f"  Team qualities — to current:       {len(team_q_to)}")
print(f"  Team z-scores — from projected:    {len(team_z_from)}")
print(f"  Team z-scores — to current:        {len(team_z_to)}")
print(f"  TOTAL:                             {len(structural)+len(twelve_qs)+len(team_q_from)+len(team_q_to)+len(team_z_from)+len(team_z_to)}")

print(f"\nAll columns:")
for i, c in enumerate(cols, 1):
    print(f"  {i:3d}. {c}")

v5 shape: 18,065 rows x 120 columns

Column groups:
  Structural/ID:                     22
  Twelve Player QS (from+to):        40
  Team qualities — from projected:   7
  Team qualities — to current:       7
  Team z-scores — from projected:    22
  Team z-scores — to current:        22
  TOTAL:                             120

All columns:
    1. wy_player_id
    2. wyscout_height
    3. wyscout_weight
    4. player_season_age
    5. tm_transfer_value
    6. tm_transfer_fee
    7. from_Minutes
    8. from_competition
    9. from_position
   10. from_season
   11. from_team_id
   12. to_Minutes
   13. to_competition
   14. to_position
   15. to_season
   16. to_team_id
   17. from_comp_division
   18. from_comp_season_id
   19. from_comp_season_name
   20. to_comp_division
   21. to_comp_season_id
   22. to_comp_season_name
   23. from_Active defence
   24. from_Aerial threat
   25. from_Box threat
   26. from_Chance prevention
   27. from_Composure
   28. from_Defensive heading
   2

## 5. Save v5

In [15]:
v5.to_parquet(OUTPUT_PATH, index=False)
size_mb = os.path.getsize(OUTPUT_PATH) / 1e6

print(f"Saved: {OUTPUT_PATH}")
print(f"Size:  {size_mb:.1f} MB")
print(f"Shape: {v5.shape[0]:,} rows x {v5.shape[1]} columns")
print(f"\nChanges from v4:")
print(f"  DROPPED: 148 raw team stats (from_team_stats_*, to_team_stats_*)")
print(f"  ADDED:   from_q_proj_* (7)  — origin team projected qualities")
print(f"  ADDED:   to_q_* (7)         — destination team current qualities")
print(f"  ADDED:   from_z_proj_* (21) — origin team projected z-scores")
print(f"  ADDED:   to_z_* (21)        — destination team current z-scores")

Saved: ../../../thesis_data/processed_data/thesis_model_dataset/within_league_transfers_v5.parquet
Size:  10.4 MB
Shape: 18,065 rows x 120 columns

Changes from v4:
  DROPPED: 148 raw team stats (from_team_stats_*, to_team_stats_*)
  ADDED:   from_q_proj_* (7)  — origin team projected qualities
  ADDED:   to_q_* (7)         — destination team current qualities
  ADDED:   from_z_proj_* (21) — origin team projected z-scores
  ADDED:   to_z_* (21)        — destination team current z-scores
